# 00 · ESPN weekly pull

The weekly version of `00_espn_pull`. It pulls only the current season, works out the last finished week on its own, and swaps this season's rows in the CSVs for fresh ones. Past seasons stay exactly as `00_espn_pull` saved them, so run that full pull once first.

A public league needs no cookies. If the league ever goes private, add `ESPN_S2` and `SWID` as secrets in the GitHub repo (Settings, then Secrets and variables, then Actions) and this picks them up. They never go in the notebook.

In [ ]:
from pathlib import Path
import os, time
import pandas as pd
from espn_api.football import League

LEAGUE_ID = 35924907
THROUGH_WEEK = None        # None uses the last finished week

DATA = Path("data")
PREFIX = f"espn_{LEAGUE_ID}_"
if not (DATA / f"{PREFIX}teams.csv").exists():
    raise FileNotFoundError("Run 00_espn_pull once first. This notebook only refreshes the current season.")

## Connect

ESPN's current week is the one being played now, so the last finished week is the one before it (or the final week once the season is over).

In [ ]:
# The NFL season that's running: a season starts in September and ends in early January
today = pd.Timestamp.today()
season_now = today.year if today.month >= 8 else today.year - 1
league = League(league_id=LEAGUE_ID, year=season_now,
                espn_s2=os.environ.get("ESPN_S2") or None, swid=os.environ.get("SWID") or None)
SEASON = league.year
last_week = THROUGH_WEEK or min(max(league.current_week - 1, 1), league.finalScoringPeriod)
print(league.settings.name, "·", SEASON, "· pulling weeks 1 to", last_week)

## Helpers

The same translations as `00_espn_pull`: ESPN's FLEX, bench, and defense names become the site's.

In [ ]:
SLOT = {"RB/WR/TE": "FLEX", "BE": "BN", "D/ST": "DEF", "OP": "SUPERFLEX", "WR/TE": "WR/TE", "RB/WR": "RB/WR"}
POS = {"D/ST": "DEF"}

def manager(team):
    owners = getattr(team, "owners", None) or []
    names = []
    for o in owners:
        if isinstance(o, dict):
            names.append(f"{o.get('firstName', '')} {o.get('lastName', '')}".strip() or o.get("displayName", ""))
        else:
            names.append(str(o))
    return " & ".join(n for n in names if n)

def team_id(team):
    return getattr(team, "team_id", None)

## Pull this season

Teams, every week's scores and lineups, the draft, and every add, drop, and trade.

In [ ]:
teams_rows, sched_rows, lineup_rows, draft_rows, move_rows = [], [], [], [], []
lg = league

for t in lg.teams:
    teams_rows.append(dict(season=SEASON, team_id=t.team_id, team_name=t.team_name, abbrev=t.team_abbrev, manager=manager(t),
                           wins=t.wins, losses=t.losses, ties=t.ties, points_for=t.points_for, points_against=t.points_against,
                           seed=t.standing, final_place=(t.final_standing or None) if last_week >= lg.finalScoringPeriod else None))

cache = {}
for week in range(1, last_week + 1):
    for g in lg.box_scores(week, player_team_cache=cache):
        sched_rows.append(dict(season=SEASON, week=week, home_team_id=team_id(g.home_team), away_team_id=team_id(g.away_team),
                               home_score=g.home_score, away_score=g.away_score, bracket=getattr(g, "matchup_type", "NONE")))
        for side, lineup in (("home", g.home_lineup), ("away", g.away_lineup)):
            tid = team_id(getattr(g, f"{side}_team"))
            if tid is None:
                continue
            for p in lineup:
                lineup_rows.append(dict(season=SEASON, week=week, team_id=tid, player_id=p.playerId, player_name=p.name,
                                        nfl_team=p.proTeam, position=POS.get(p.position, p.position),
                                        slot=SLOT.get(p.slot_position, p.slot_position), points=p.points,
                                        projected=p.projected_points, injury=getattr(p, "injuryStatus", "")))
    time.sleep(0.3)

picks = lg.draft or []
ids = [p.playerId for p in picks]; pos_of = {}
for i in range(0, len(ids), 50):
    found = lg.player_info(playerId=ids[i:i + 50]) or []
    found = found if isinstance(found, list) else [found]
    pos_of.update({pl.playerId: POS.get(pl.position, pl.position) for pl in found})
for p in picks:
    draft_rows.append(dict(season=SEASON, round=p.round_num, round_pick=p.round_pick,
                           overall=(p.round_num - 1) * lg.settings.team_count + p.round_pick, team_id=team_id(p.team),
                           player_id=p.playerId, player_name=p.playerName, position=pos_of.get(p.playerId, ""), keeper=bool(p.keeper_status)))

offset, n = 0, 0
while True:
    batch = lg.recent_activity(size=25, offset=offset)
    if not batch:
        break
    for a in batch:
        n += 1
        for team, action, player, bid in a.actions:
            move_rows.append(dict(season=SEASON, move_id=f"{SEASON}-{n}", date=pd.to_datetime(a.date, unit="ms"),
                                  team_id=team_id(team), action=action, player_id=getattr(player, "playerId", player),
                                  player_name=getattr(player, "name", ""),
                                  position=POS.get(getattr(player, "position", ""), getattr(player, "position", "")), bid=bid))
    offset += 25
    time.sleep(0.3)
print(len(sched_rows), "games ·", len(lineup_rows), "lineup rows ·", len(draft_rows), "picks ·", n, "transactions")

## Swap this season into the files

Each file keeps every past season and gets this season's rows replaced. The settings file is rewritten from the current season.

In [ ]:
new = {"teams": pd.DataFrame(teams_rows), "schedule": pd.DataFrame(sched_rows), "lineups": pd.DataFrame(lineup_rows),
       "draft": pd.DataFrame(draft_rows), "transactions": pd.DataFrame(move_rows)}
for name, df in new.items():
    f = DATA / f"{PREFIX}{name}.csv"
    old = pd.read_csv(f)
    out = pd.concat([old[old.season != SEASON], df], ignore_index=True)
    if name == "teams":
        out = out.astype({"final_place": "Int64"})
    out.to_csv(f, index=False)
    print(f"{name}: {len(old[old.season != SEASON]):,} past rows kept, {len(df):,} for {SEASON}")

s = lg.settings
rows = [("name", s.name), ("teams", s.team_count), ("regular_season_weeks", s.reg_season_count), ("playoff_teams", s.playoff_team_count),
        ("playoff_matchup_length", getattr(s, "playoff_matchup_period_length", "")), ("scoring_type", getattr(s, "scoring_type", "")),
        ("tie_rule", s.tie_rule), ("playoff_seed_tie_rule", s.playoff_seed_tie_rule), ("keepers", s.keeper_count), ("faab", s.faab)]
rows += [(f"slot_{SLOT.get(k, k)}", v) for k, v in s.position_slot_counts.items() if v]
rows += [(f"score_{f['abbr']}", f["points"]) for f in s.scoring_format]
pd.DataFrame(rows, columns=["setting", "value"]).to_csv(DATA / f"{PREFIX}settings.csv", index=False)
print("Through week", last_week)